In [ ]:
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("MNIST Dataset")
print("Tracking URI:", mlflow.get_tracking_uri())

In [ ]:
X, y = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state = 42)

def train_and_evaluate(hidden_layer_sizes=(128,),alpha=0.0001,learning_rate_init=0.001,batch_size=64):
    model = MLPClassifier(hidden_layer_sizes=hidden_layer_sizes,alpha=alpha,learning_rate_init=learning_rate_init,
                          batch_size=batch_size,max_iter=20,random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_train)
    train_acc = accuracy_score(y_train, preds)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="macro")
    return model, train_acc, acc, f1

_, train_acc, acc, f1 = train_and_evaluate()
print(f"accuracy={acc:.4f} train_accuracy={train_acc:.4f} f1_macro={f1:.4f} " )

In [ ]:
def train_and_log(alpha = 0.0001, learning_rate = 0.001, batch_size = 64, run_name=None):
    with mlflow.start_run(run_name=run_name):

        mlflow.log_param("Alpha", alpha)
        mlflow.log_param("Learning Rate", learning_rate)
        mlflow.log_param("Batch Size", batch_size)

        model, train_acc, acc, f1 = train_and_evaluate(alpha = alpha, learning_rate_init=learning_rate, batch_size=batch_size)

        # --- metrics (at least 2) ---
        mlflow.log_metric("training accuracy", train_acc)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_macro", f1)

        mlflow.set_tag("Assignment 1",'Test Run')
        mlflow.sklearn.log_model(model,
        name="model",
        serialization_format="skops",
        skops_trusted_types=[
            "sklearn.neural_network._stochastic_optimizers.AdamOptimizer"
        ])

        run_id = mlflow.active_run().info.run_id
        print(f"Logged run {run_id}  |  acc={acc:.4f}  f1={f1:.4f} train_acc={train_acc:.4f}")
        return run_id

In [ ]:
import warnings
warnings.filterwarnings('ignore')
sweep_run_ids = []
for lr in [0.001,0.002,0.000001]:
    for alpha in [0.001,0.01]:
        for batch_size in [32,64]:
            rid = train_and_log(alpha = alpha, learning_rate = lr, batch_size = batch_size,
                                run_name=f"rf-alpha-{alpha}-learning_rate-{lr}-batch_size-{batch_size}")
            sweep_run_ids.append(rid)

print("Sweep run IDs:", sweep_run_ids)